# Evaluate your AI agent with Weave

This notebook shows how to evaluate an AI agent with Weave using the new
**Agents / Conversations** workflow — the agent is traced as a conversation of
turns and tool calls (OpenTelemetry spans), and you score it with
`weave.EvaluationLogger`.

You'll:
1. Build and trace a small customer-support agent.
2. Score single-turn **task completion** with an LLM judge.
3. Compare two agent versions.
4. Score a **multi-turn** conversation.

The agent runs on **Claude Sonnet**; the judge runs on **Claude Opus** — using a
different model to grade than to act is good evaluation practice.

> This notebook makes real LLM calls (agent + judge across several tasks), so it
> will incur provider costs.

## Setup

Install dependencies and provide your API keys.

In [ ]:
!pip install -qU weave anthropic

In [ ]:
import os, getpass

# In Colab, prefer the Secrets panel (key icon) named WANDB_API_KEY / ANTHROPIC_API_KEY.
try:
    from google.colab import userdata
    for k in ["WANDB_API_KEY", "ANTHROPIC_API_KEY"]:
        os.environ.setdefault(k, userdata.get(k))
except Exception:
    for k in ["WANDB_API_KEY", "ANTHROPIC_API_KEY"]:
        if not os.environ.get(k):
            os.environ[k] = getpass.getpass(f"{k}: ")

In [ ]:
import weave

weave.init(
    "agent-eval-tutorial",  # your Weave project
    # Turn off implicit patching of manual Weave Ops tracing (Traces tab) — the
    # bare Anthropic SDK integration would log each LLM call as a legacy Op/Call
    # in parallel with our hand-rolled Conversation SDK spans, duplicating it.
    settings={"implicitly_patch_integrations": False},
)

## 1. Build and trace the agent

A minimal customer-support agent with two tools (`lookup_order`, `issue_refund`)
and a policy: refunds are only allowed within 30 days.

The Weave-specific part is the **Conversation SDK** instrumentation inside
`run_agent_turn`: `start_conversation` → `start_turn` → `start_llm` /
`start_tool`, with `llm.record(...)` capturing the model call. Everything else
(the tool backend, the Anthropic tool-use loop, message conversion) is ordinary
agent plumbing.

In [ ]:
import json
import anthropic

# Agent and judge deliberately use different models (good eval practice).
AGENT_MODEL = "claude-sonnet-5"
_client = anthropic.Anthropic()

# --- Mock support backend (tools the agent can call) ---
_ORDERS = {
    "A1001": {"item": "Wireless headphones", "amount": 79.99, "days_since_delivery": 5},
    "A1002": {"item": "Standing desk", "amount": 320.00, "days_since_delivery": 60},
}
REFUND_WINDOW_DAYS = 30

def lookup_order(order_id):
    order = _ORDERS.get(order_id)
    return {"error": f"order {order_id} not found"} if order is None else {"order_id": order_id, **order}

def issue_refund(order_id, amount):
    order = _ORDERS.get(order_id)
    if order is None:
        return {"refunded": False, "reason": "order not found"}
    if order["days_since_delivery"] > REFUND_WINDOW_DAYS:
        return {"refunded": False, "reason": "outside 30-day refund window"}
    return {"refunded": True, "order_id": order_id, "amount": amount}

_DISPATCH = {"lookup_order": lookup_order, "issue_refund": issue_refund}

TOOLS = [
    {"name": "lookup_order", "description": "Look up an order by its id.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
    {"name": "issue_refund", "description": "Issue a refund for an order.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"},
                      "amount": {"type": "number"}}, "required": ["order_id", "amount"]}},
]

SYSTEM = ("You are a customer-support agent. Refunds are only allowed within the "
          "30-day window. Always look up the order before issuing a refund. If a "
          "refund is not allowed, explain why politely.")

In [ ]:
from weave.conversation import Message, ToolCallPart, Usage

# --- Convert Anthropic message blocks <-> weave.Message (agent plumbing) ---
def _assistant_out(content):
    text = "".join(b.text for b in content if b.type == "text")
    tcs = [ToolCallPart(id=b.id, name=b.name, arguments=json.dumps(b.input))
           for b in content if b.type == "tool_use"]
    return Message.assistant(text, tool_calls=tcs or None)

def _to_weave_inputs(messages):
    out = [Message.system(SYSTEM)]
    for m in messages:
        role, content = m["role"], m["content"]
        if role == "user" and isinstance(content, str):
            out.append(Message.user(content))
        elif role == "user":
            out += [Message.tool_result(b["tool_use_id"], b["content"]) for b in content]
        elif role == "assistant":
            out.append(Message.assistant(content) if isinstance(content, str) else _assistant_out(content))
    return out

def run_agent_turn(conversation, user_message, model=AGENT_MODEL, history=None):
    # Run one user turn to completion; return (final_reply, full_message_log).
    messages = list(history or [])
    messages.append({"role": "user", "content": user_message})

    with conversation.start_turn(user_message=user_message) as turn:
        while True:
            with turn.start_llm(model=model, provider_name="anthropic") as llm:
                resp = _client.messages.create(
                    model=model, max_tokens=2048, system=SYSTEM, tools=TOOLS, messages=messages,
                )
                llm.record(
                    input_messages=_to_weave_inputs(messages),
                    output_messages=[_assistant_out(resp.content)],
                    usage=Usage(input_tokens=resp.usage.input_tokens,
                                output_tokens=resp.usage.output_tokens),
                    response_id=resp.id, response_model=resp.model,
                    finish_reasons=[resp.stop_reason],
                )
            messages.append({"role": "assistant", "content": resp.content})

            if resp.stop_reason != "tool_use":
                return "".join(b.text for b in resp.content if b.type == "text"), messages

            tool_results = []
            for block in resp.content:
                if block.type != "tool_use":
                    continue
                with turn.start_tool(name=block.name, arguments=json.dumps(block.input),
                                     tool_call_id=block.id) as tool:
                    tool.result = _DISPATCH[block.name](**block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                     "content": json.dumps(tool.result)})
            messages.append({"role": "user", "content": tool_results})

In [ ]:
import uuid
from weave.conversation import start_conversation

with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
    reply, _ = run_agent_turn(conv, "I'd like a refund for order A1001, please.")
print(reply)

Open the printed Weave link and find the **Agents** view: the conversation
appears as a turn with the model call and tool calls nested inside it.

> **Prefer auto-instrumentation?** If you build your agent with an
> agent-framework integration (e.g. the Claude Agent SDK or OpenAI Agents),
> Weave emits these same Agents spans automatically — leave implicit patching on
> and skip the manual `start_*` calls. Auto-patching the *bare* provider SDK,
> by contrast, produces legacy Ops/Calls, not Agents spans.

## 2. Score task completion (single turn)

**Task completion** = an LLM judge decides whether the agent achieved the goal,
judged against the task's success criteria from the whole transcript — not a
golden-string match.

In [ ]:
JUDGE_MODEL = "claude-opus-4-8"  # a different, stronger model than the agent

def _extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        text = text[4:] if text.startswith("json") else text
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

def judge_task_completion(task, transcript):
    prompt = (
        "You are evaluating whether a customer-support agent completed the "
        "user's task. Judge the whole transcript against the success criteria "
        "— reward the correct OUTCOME, not just a polite reply.\n\n"
        f"USER REQUEST: {task['user_request']}\n"
        f"SUCCESS CRITERIA: {task['success_criteria']}\n\n"
        f"AGENT TRANSCRIPT:\n{transcript}\n\n"
        'Respond with ONLY a JSON object: {"passed": <true|false>, '
        '"reason": "<one sentence>"}. Set passed=true only if the success '
        "criteria are met."
    )
    resp = _client.messages.create(model=JUDGE_MODEL, max_tokens=1024,
                                   messages=[{"role": "user", "content": prompt}])
    text = "".join(b.text for b in resp.content if b.type == "text")
    return _extract_json(text)

def render_transcript(messages, final_reply):
    lines = []
    for m in messages:
        role, content = m["role"], m["content"]
        if isinstance(content, str):
            lines.append(f"{role.capitalize()}: {content}"); continue
        for b in content:
            if isinstance(b, dict):
                lines.append(f"Tool result: {b['content']}")
            elif b.type == "text" and b.text:
                lines.append(f"Agent: {b.text}")
            elif b.type == "tool_use":
                lines.append(f"Agent called {b.name}({json.dumps(b.input)})")
    lines.append(f"Final reply: {final_reply}")
    return "\n".join(lines)

Now drive the evaluation with `EvaluationLogger`. The key move: run the agent
**inside** `log_prediction(...)`, so each trial's agent transcript links to the
evaluation result.

In [ ]:
TASKS = [
    {"task_id": "refund-eligible",
     "user_request": "I'd like a refund for order A1001, please.",
     "success_criteria": "Agent looks up the order and issues the refund (within the 30-day window)."},
    {"task_id": "refund-too-late",
     "user_request": "Please refund my order A1002.",
     "success_criteria": "Agent looks up the order and politely declines (outside the 30-day window); must NOT issue a refund."},
    {"task_id": "unknown-order",
     "user_request": "I want a refund for order Z9999.",
     "success_criteria": "Agent reports the order cannot be found and does not issue a refund."},
]

ev = weave.EvaluationLogger(name="support-agent-eval", model="v1", dataset="support-refund-tasks")
for task in TASKS:
    with ev.log_prediction(inputs=task) as pred:
        with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
            reply, messages = run_agent_turn(conv, task["user_request"])
        verdict = judge_task_completion(task, render_transcript(messages, reply))
        pred.output = reply
        pred.log_score("task_completion", verdict)
        print(f"[{task['task_id']}] passed={verdict['passed']}: {verdict['reason']}")
ev.log_summary()
print("Evaluation:", ev.ui_url)

Open the evaluation link: each row is a task, with its task-completion score,
output, latency, and cost — and a link to the full agent trace for that trial.

**A scorecard, not one number.** Task completion is one signal. Add more with
extra `pred.log_score(...)` calls in the same block — e.g. `tool_call_correct`
or `instruction_following`.

## 3. Organize and compare evaluations

Run the same evaluation for a second agent version by changing the `model` label
(e.g. `EvaluationLogger(model="v2", ...)` after editing the agent's prompt or
switching its model). Weave lays the two runs **side by side** in the comparison
view, so you can read task-completion rate, tool-call correctness, latency, and
cost for v1 vs v2 — the baseline-relative question ("did the candidate do as well
as the baseline?"). Every row still links to its agent transcript, so a
regression is one click from the failing conversation.

## 4. Score a multi-turn conversation

To evaluate a turn *in context*, load a fixed conversation history, append one
new user turn, and score how the agent handles it — i.e. *"given the
conversation state so far, does the agent handle the next turn well?"*

Below, the order id is only mentioned in the prior turns; a good agent should use
that context instead of asking again.

In [ ]:
MULTI_TURN_TASKS = [
    {"task_id": "context-refund",
     "conversation_history": [
         {"role": "user", "content": "Hi, can you check the status of my order A1001?"},
         {"role": "assistant", "content": "Your order A1001 (Wireless headphones) was delivered 5 days ago."},
     ],
     "next_user_message": "Thanks. Actually, I'd like to return it for a refund.",
     "success_criteria": "Using the prior context that the order is A1001, the agent issues the refund (within 30 days) WITHOUT asking the user to repeat the order id."},
    {"task_id": "context-decline",
     "conversation_history": [
         {"role": "user", "content": "Can you look up order A1002 for me?"},
         {"role": "assistant", "content": "Order A1002 (Standing desk) was delivered 60 days ago."},
     ],
     "next_user_message": "Okay, I'd like a refund for it.",
     "success_criteria": "Using the prior context that A1002 was delivered 60 days ago, the agent politely declines as outside the 30-day window and does NOT issue a refund."},
]

ev = weave.EvaluationLogger(name="support-agent-multiturn-eval", model="v1", dataset="support-multiturn-tasks")
for task in MULTI_TURN_TASKS:
    with ev.log_prediction(inputs=task) as pred:
        with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
            reply, messages = run_agent_turn(conv, task["next_user_message"],
                                             history=task["conversation_history"])
        judge_task = {"user_request": task["next_user_message"], "success_criteria": task["success_criteria"]}
        verdict = judge_task_completion(judge_task, render_transcript(messages, reply))
        pred.output = reply
        pred.log_score("task_completion", verdict)
        print(f"[{task['task_id']}] passed={verdict['passed']}: {verdict['reason']}")
ev.log_summary()
print("Evaluation:", ev.ui_url)

> **Scope.** This scores the *next* turn against a fixed history — the practical
> offline approach. Measuring a full multi-turn task end-to-end (where the agent
> drives the whole session) requires live A/B testing in production and is out of
> scope here.

## 5. Extend your scorers

Real agents need a scorecard covering both:
- **Functional:** tool-call correctness, instruction-following, recovery from tool errors.
- **Non-functional:** safety / refusal behavior, latency, cost, hallucinated-tool use.

Add each as another `pred.log_score(name, value)` inside the prediction block.

## Recap
You traced an agent as a conversation, scored task completion for single and
multi-turn interactions, and compared versions — all linked back to the agent
transcripts. For **online** evaluation (automatically scoring every agent trace
in production), see Weave's production monitoring docs.